# Autoresearch metric-discovery — run analysis
Mirrors Karpathy's `analysis.ipynb`: reads `results.tsv`, tallies keep/discard/crash, plots the validation-score frontier over experiments, and lists the improvements that stuck. Higher `val_score` is better (downside-forecast rank corr).

The locked-TEST result is NOT here — it is produced once by `final_eval.py` after the loop.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
df = pd.read_csv('results.tsv', sep='\t')
df['val_score'] = pd.to_numeric(df['val_score'], errors='coerce')
df['status'] = df['status'].str.upper()
print(f'{len(df)} experiments'); print(df.columns.tolist()); df.head()

In [ ]:
# Outcome tally + keep rate (excluding crashes)
counts = df['status'].value_counts(); print(counts)
decided = counts.get('KEEP',0) + counts.get('DISCARD',0)
print(f'keep rate: {counts.get("KEEP",0)/decided:.1%}' if decided else 'n/a')

In [ ]:
# Progress plot: val_score over experiments, with running-best frontier
ok = df[df['status'] != 'CRASH'].reset_index(drop=True)
kept = ok[ok['status'] == 'KEEP']; disc = ok[ok['status'] == 'DISCARD']
frontier = ok['val_score'].cummax()
plt.figure(figsize=(9,5))
plt.scatter(disc.index, disc['val_score'], c='lightgray', label='discarded')
plt.scatter(kept.index, kept['val_score'], c='green', label='kept', zorder=3)
plt.step(ok.index, frontier, where='post', c='green', alpha=0.5, label='best-so-far (frontier)')
plt.xlabel('experiment'); plt.ylabel('val_score (higher better)'); plt.legend(); plt.title('Val score over experiments')
plt.tight_layout(); plt.savefig('progress.png', dpi=120); plt.show()

In [ ]:
# Summary + top hits (each kept improvement vs the previous kept state)
base = ok['val_score'].iloc[0]; best = ok['val_score'].max()
print(f'baseline {base:.4f} -> best {best:.4f}  (+{best-base:.4f}, {(best-base)/abs(base):.1%})')
k = kept.copy(); k['delta'] = k['val_score'].diff().fillna(k['val_score'] - base)
print('\nImprovements that stuck (by size):')
print(k.sort_values('delta', ascending=False)[['val_score','delta','description']].to_string(index=False))